# Lesson 5.3 — Compliance: AWS Config, Security Hub, GuardDuty & Inspector

**Level: Intermediate** | **Exam:** SOA-C03 Domain 4, Task 4.2 (Skills 4.2.4, 4.2.5)

## Learning Objectives
- Configure AWS Config rules for continuous compliance monitoring
- Centralize security findings with AWS Security Hub
- Interpret Amazon GuardDuty findings and respond to threats
- Use Amazon Inspector for vulnerability assessments on EC2 and containers
- Build an automated compliance remediation workflow

_Reference: [AWS Config](https://docs.aws.amazon.com/config/latest/developerguide/) | [Security Hub](https://docs.aws.amazon.com/securityhub/latest/userguide/) | [GuardDuty](https://docs.aws.amazon.com/guardduty/latest/ug/)_

## The Compliance Stack

```
AWS Security Services:
┌──────────────────────────────────────────────────────────────┐
│  AWS Security Hub  (aggregates findings from all services)   │
├─────────────┬───────────────┬──────────────┬────────────────-┤
│ GuardDuty   │ AWS Config    │  Inspector   │ Macie           │
│ (threat)    │ (compliance)  │ (vulns)      │ (data security) │
└─────────────┴───────────────┴──────────────┴─────────────────┘
```

## AWS Config
- **Continuously records** resource configurations and changes
- **Config Rules**: Check if resources comply with rules (managed or custom Lambda)
- **Conformance Packs**: Collection of rules for a standard (e.g., CIS Benchmarks, HIPAA, PCI-DSS)
- **Remediation Actions**: SSM Automation documents triggered when rules are non-compliant

**Useful managed rules:**
- `encrypted-volumes` — EBS volumes must be encrypted
- `restricted-ssh` — No security groups with port 22 open to 0.0.0.0/0
- `s3-bucket-public-read-prohibited` — S3 buckets must not allow public read
- `root-account-mfa-enabled` — Root account must have MFA
- `cloudtrail-enabled` — CloudTrail must be enabled

## Amazon GuardDuty
- Machine learning–based threat detection
- Analyzes: **VPC Flow Logs**, **CloudTrail**, **DNS logs**, **S3 data events**
- Finding types: Reconnaissance, Backdoor, CryptoCurrency mining, Exfiltration
- No infrastructure to manage — just enable it
- Costs based on data processed

**Response to GuardDuty findings:**
```
GuardDuty Finding → EventBridge rule → Lambda → Remediate
  e.g., UnauthorizedAccess:IAMUser/InstanceCredentialExfiltration
  → Disable IAM key, isolate instance, notify security team
```

## Amazon Inspector
- Automated vulnerability scanning for **EC2 instances** and **ECR container images**
- Findings prioritized by **CVSS score** + **exploitability**
- Integrates with Security Hub for centralized view

## AWS Macie
- Discovers and protects **sensitive data in S3** (PII, credentials, financial data)
- Uses ML to classify data automatically

In [ ]:
import boto3, json

config = boto3.client('config', region_name='us-east-1')
guardduty = boto3.client('guardduty', region_name='us-east-1')
security_hub = boto3.client('securityhub', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# List AWS Config managed rules and their compliance
try:
    rules = config.describe_config_rules()['ConfigRules']
    if rules:
        print(f'Config Rules ({len(rules)}):')
        # Get compliance for all rules at once
        compliance = config.describe_compliance_by_config_rule(
            ConfigRuleNames=[r['ConfigRuleName'] for r in rules[:25]]
        )['ComplianceByConfigRules']
        comp_map = {c['ConfigRuleName']: c['Compliance']['ComplianceType'] for c in compliance}
        
        for rule in rules[:10]:
            name = rule['ConfigRuleName']
            status = comp_map.get(name, 'UNKNOWN')
            icon = '✓' if status == 'COMPLIANT' else ('✗' if status == 'NON_COMPLIANT' else '?')
            print(f'  {icon} {name:50} {status}')
    else:
        print('No Config rules found. Enable AWS Config and add rules.')
        print('\nUseful rules to enable:')
        rules_to_enable = [
            'encrypted-volumes',
            'restricted-ssh', 
            's3-bucket-public-read-prohibited',
            'cloudtrail-enabled',
            'root-account-mfa-enabled',
            'iam-password-policy'
        ]
        for r in rules_to_enable:
            print(f'  {r}')
except Exception as e:
    print(f'Config: {e} (Is AWS Config enabled in your account?)')

In [ ]:
# Check GuardDuty status and findings
try:
    detectors = guardduty.list_detectors()['DetectorIds']
    if detectors:
        detector_id = detectors[0]
        detector = guardduty.get_detector(DetectorId=detector_id)
        print(f'GuardDuty Detector: {detector_id}')
        print(f'Status: {detector["Status"]}')
        print(f'Finding publishing frequency: {detector["FindingPublishingFrequency"]}')
        
        # Get sample findings summary
        findings_response = guardduty.list_findings(DetectorId=detector_id, MaxResults=5)
        finding_ids = findings_response['FindingIds']
        if finding_ids:
            findings = guardduty.get_findings(DetectorId=detector_id, FindingIds=finding_ids)['Findings']
            print(f'\nLatest Findings ({len(finding_ids)}):')
            for f in findings:
                print(f'  [{f["Severity"]:.1f}] {f["Type"]:50} Region: {f["Region"]}')
        else:
            print('No active GuardDuty findings (account may be clean, or GuardDuty recently enabled)')
    else:
        print('GuardDuty not enabled. Enable with:')
        print("guardduty.create_detector(Enable=True, FindingPublishingFrequency='FIFTEEN_MINUTES')")
except Exception as e:
    print(f'GuardDuty: {e}')

In [ ]:
# Security Hub — aggregated security findings
try:
    hub = security_hub.describe_hub()
    print(f'Security Hub ARN: {hub["HubArn"]}')
    
    # Get finding summary by severity
    findings = security_hub.get_findings(
        Filters={
            'RecordState': [{'Value': 'ACTIVE', 'Comparison': 'EQUALS'}],
            'WorkflowStatus': [{'Value': 'NEW', 'Comparison': 'EQUALS'}]
        },
        MaxResults=10
    )['Findings']
    
    severity_counts = {}
    for f in findings:
        sev = f.get('Severity', {}).get('Label', 'UNKNOWN')
        severity_counts[sev] = severity_counts.get(sev, 0) + 1
    
    print(f'\nNew Findings by Severity:')
    for sev in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW', 'INFORMATIONAL']:
        count = severity_counts.get(sev, 0)
        bar = '#' * min(count, 20)
        print(f'  {sev:15} {count:4} {bar}')
except security_hub.exceptions.InvalidAccessException:
    print('Security Hub not enabled. Enable with:')
    print('security_hub.enable_security_hub(EnableDefaultStandards=True)')
except Exception as e:
    print(f'Security Hub: {e}')

## Exercise

In [ ]:
# TODO: Create an AWS Config rule that:
# 1. Checks that all S3 buckets have server-side encryption enabled
# 2. Uses the managed rule 's3-bucket-server-side-encryption-enabled'
# 3. Sets up auto-remediation using the SSM Automation document 'AWS-EnableS3BucketEncryption'
# Note: Config must be enabled in your account first

import boto3
config = boto3.client('config', region_name='us-east-1')

try:
    # YOUR CODE HERE
    config.put_config_rule(
        ConfigRule={
            'ConfigRuleName': 'cloudops-s3-encryption-required',
            # TODO: Add Source (managed rule), Scope, etc.
        }
    )
except Exception as e:
    print(f'Config rule: {e}')

## Summary
- **AWS Config**: Continuous configuration recording + rules for compliance. Use conformance packs for standards.
- **GuardDuty**: ML threat detection from VPC Flow Logs, CloudTrail, DNS. Always-on with no infrastructure.
- **Security Hub**: Aggregates findings from GuardDuty, Inspector, Macie, Config, and third-party tools.
- **Inspector**: Automated CVE scanning for EC2 and ECR container images.
- Config remediation = Config Rule (detect) + SSM Automation (fix) → automated compliance loop.

---
**Next Phase:** [Phase 6 — Networking & Content Delivery](../phase-6-networking/lesson-6.1-vpc-fundamentals.ipynb)